#### Step 1 - Import libraries and inspect data

In [4]:
import pandas as pd
import re

df = pd.read_csv('../data/Attendance_data_raw.csv')

df.info()
df.head()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899 entries, 0 to 898
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Unnamed: 0     899 non-null    int64 
 1   Att. Date      899 non-null    object
 2   InTime         502 non-null    object
 3   OutTime        502 non-null    object
 4   Shift          899 non-null    object
 5   S. InTime      899 non-null    object
 6   S. OutTime     899 non-null    object
 7   Work Dur.      899 non-null    object
 8   OT             899 non-null    object
 9   Tot. Dur.      899 non-null    object
 10  LateBy         899 non-null    object
 11  EarlyGoingBy   899 non-null    object
 12  Status         899 non-null    object
 13  Punch Records  502 non-null    object
 14  E_Name         899 non-null    object
dtypes: int64(1), object(14)
memory usage: 105.5+ KB


Unnamed: 0         0
Att. Date          0
InTime           397
OutTime          397
Shift              0
S. InTime          0
S. OutTime         0
Work Dur.          0
OT                 0
Tot. Dur.          0
LateBy             0
EarlyGoingBy       0
Status             0
Punch Records    397
E_Name             0
dtype: int64

In [5]:
df = df.drop(columns=['Unnamed: 0'])

In [11]:
# strip any trailing whitespaces
df['Status'] = df['Status'].str.strip()
df['Shift'] = df['Shift'].str.strip()

df.head(20)

,Att. Date,InTime,OutTime,Shift,S. InTime,S. OutTime,Work Dur.,OT,Tot. Dur.,LateBy,EarlyGoingBy,Status,Punch Records,E_Name
0,1-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,WeeklyOff,NaN,Aarav
1,2-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav
2,3-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav
3,4-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav
4,5-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav
5,6-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav
6,7-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav
7,8-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,WeeklyOff,NaN,Aarav
8,9-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav
9,10-Mar-25,NaN,NaN,GS,9:30,18:30,0:00,0:00,0:00,0:00,0:00,Absent,NaN,Aarav


In [14]:
df['Att. Date'] = pd.to_datetime(df['Att. Date'], format='%d-%b-%y')

# verify
df['Att. Date'].dtype
df['Att. Date'].head()

0   2025-03-01
1   2025-03-02
2   2025-03-03
3   2025-03-04
4   2025-03-05
Name: Att. Date, dtype: datetime64[ns]

In [15]:
time_cols =  ['InTime', 'OutTime', 'S. InTime', 'S. OutTime',
                'Work Dur.', 'OT', 'Tot. Dur.', 'LateBy', 'EarlyGoingBy']

for col in time_cols:
    df[col] = pd.to_timedelta(df[col].apply(
        lambda x: f"00:{x}" if pd.notna(x) and x != '0:00' else x
    ), errors='coerce')

#### Parse the Punch Records Column

In [23]:
def parse_valid_punches(raw):
    if pd.isna(raw):
        return []
    # Match pattern: time:(TYPE) where TYPE is IN or OUT
    matches = re.findall(r'(\d{1,2}:\d{2}):[^(]*\((\w+)\)', raw)
    return [(time, direction) for time, direction in matches
            if direction in ('IN', 'OUT')]

def validate_punch_sequence(punches):
    issues = []

    if not punches:
        return ['No valid punches']

    directions = [p[1] for p in punches]

    # Should start with IN
    if directions[0] != 'IN':
        issues.append('First punch is not IN')

    # Should end with OUT
    if directions[-1] != 'OUT':
        issues.append('Last punch is not OUT - possible missing exit')

    # No two consecutive same direction (door malfunction)
    for i in range(len(directions) - 1):
        if directions[i] == directions [i+1]:
            issues.append(f'Consecutive {directions[i]} punches at index {i} - door malfunction or tailgate')
    return issues if issues else ['Clean']

# Apply both functions
df['Parsed_punches'] = df['Punch Records'].apply(parse_valid_punches)
df['Punch_issues'] = df['Parsed_punches'].apply(validate_punch_sequence)
df['Punch_flag'] = df['Punch_issues'].apply(
    lambda x: 'Clean' if x == ['Clean'] else ' | '.join(x)
)

print(df[df['Punch_flag'] != 'Clean']['Punch_flag'].value_counts())


Punch_flag
No valid punches                                                                                                                                                                                           397
Consecutive IN punches at index 0 - door malfunction or tailgate                                                                                                                                            21
Consecutive IN punches at index 0 - door malfunction or tailgate | Consecutive OUT punches at index 2 - door malfunction or tailgate                                                                        11
Last punch is not OUT - possible missing exit                                                                                                                                                               10
Consecutive IN punches at index 2 - door malfunction or tailgate                                                                                                 

In [25]:
def categorize_status(status):
    if 'WeeklyOff' in str(status) and 'Present' in str(status):
        return 'Weekend Present'
    elif 'WeeklyOff' in str(status):
        return 'Weekly Off'
    elif 'Present' in str(status):
        return 'Present'
    elif 'Absent' in str(status):
        return 'Absent'
    else:
        return 'Other'

df['Status_clean'] = df['Status'].apply(categorize_status)

print(df['Status_clean'].value_counts())

Status_clean
Present            432
Absent             322
Weekly Off          76
Weekend Present     69
Name: count, dtype: int64


In [26]:
def late_category(lateby):
    if pd.isna(lateby):
        return 'N/A'
    minutes = lateby.total_seconds() / 60
    if minutes > 60:
        return 'Severe - over 1 hour'
    elif minutes > 30:
        return 'Moderate - 30 to 60 min'
    elif minutes > 0:
        return 'Minor - under 30 min'
    else:
        return 'On time'

df['Late_category'] = df['LateBy'].apply(late_category)

In [28]:
# Final shape check
print('Clean dataset shape:', df.shape)
print()
print('Remaining nulls (expected for absent/off days):')
print(df[['InTime','OutTime','Punch Records']].isnull().sum())
print()
print('Status breakdown:')
print(df['Status_clean'].value_counts())
print()
print('Punch integrity:')
print(df['Punch_flag'].value_counts())

# Export cleaned dataset
df.to_csv('..\data\Attendance_data_clean.csv', index=False)
print()
print('Exported: Attendance_data_clean.csv')

Clean dataset shape: (899, 19)

Remaining nulls (expected for absent/off days):
InTime           397
OutTime          397
Punch Records    397
dtype: int64

Status breakdown:
Status_clean
Present            432
Absent             322
Weekly Off          76
Weekend Present     69
Name: count, dtype: int64

Punch integrity:
Punch_flag
No valid punches                                                                                                                                                                                           397
Clean                                                                                                                                                                                                      271
Consecutive IN punches at index 0 - door malfunction or tailgate                                                                                                                                            21
Consecutive IN punches at index 0 - door mal

In [30]:
print(df[(df['Status_clean'] == 'Present') & (df['Punch Records'].isna())].shape[0])

0


#### Simplify flags for Power BI

In [38]:
def simplify_flag(flag):
    if flag == 'No valid punches':
        return 'No data'
    elif flag == 'Clean':
        return 'Clean'
    elif 'Consecutive' in flag and 'Last punch is not OUT' in flag:
        return 'Multiple issues'
    elif 'Last punch is not OUT' in flag:
        return 'Missing exit punch'
    elif 'First punch is not IN' in flag:
        return 'Missing entry punch'
    elif 'Consecutive' in flag:
        return 'Door malfunction'
    else:
        return 'Multiple issues'

df['Punch_flag_simple'] = df['Punch_flag'].apply(simplify_flag)
print(df['Punch_flag_simple'].value_counts())

# Drop redundant column
df = df.drop(columns=['Punch_flag'])

# Re-export with this new column
df.to_csv('../data/Attendance_data_clean.csv', index=False)


Punch_flag_simple
No data                397
Clean                  271
Door malfunction       204
Multiple issues         16
Missing exit punch      10
Missing entry punch      1
Name: count, dtype: int64
